# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. {"A":2, "B":1}
Код считает количество появлении каждой буквы в листе. Функция get(с, 0) пытается извлечь значение элемента с ключом с в противном случае возращая ноль. 
2. [0, 1, 1] 
Функция просто добавляет число 1 в конец списка она вызывается два раза, но в списке уже пресутсвует ноль поэтому две однерки добавляются в конец
3. 2
После lower almaty повторяется два раза но set хранит только уникальные значения поэтому останется два города
4. {"ab": 2, "abc": 3}
len считает количество букв а одинаковый ключ ab перезаписывается поэтому второй раз он не добавится
5. Сначала 5 потом None
Функция выводит 5 но return в ней нет поэтому в x записывается None и он выводится потом


In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
csv_path = "orders.csv"
if not os.path.exists(csv_path):
    downloaded_file = os.path.join(os.path.expanduser("~"), "Downloads", "orders.csv")
    if os.path.exists(downloaded_file):
        csv_path = downloaded_file
    else:
        try:
            from google.colab import files
            files.upload()
        except ImportError:
            print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists(csv_path))


orders.csv на месте: True


In [2]:
import csv
with open(csv_path, encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")


5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [3]:
# База
def to_num(s):
    return float(s.strip().replace(" ", "").replace("\u00a0", "").replace(",", "."))

def clean_city(s):
    city = s.strip().lower()
    city_names = {
        "almaty": "Алматы",
        "astana": "Астана",
        "shymkent": "Шымкент",
        "karaganda": "Караганда",
        "aktobe": "Актобе",
        "atyrau": "Атырау"
    }
    return city_names.get(city, city.capitalize())

count_by_category = {}
for row in rows:
    category = row["category"]
    count_by_category[category] = count_by_category.get(category, 0) + 1
print("Заказы по категориям")
for category, count in sorted(count_by_category.items(), key=lambda x: x[1], reverse=True):
    print(f"{category} | {count}")

revenue_by_category = {category: 0 for category in count_by_category}
paid_count_by_category = {category: 0 for category in count_by_category}
for row in rows:
    if row["status"] == "оплачен":
        category = row["category"]
        revenue_by_category[category] += to_num(row["amount"])
        paid_count_by_category[category] += 1
print("Категория | Выручка | Средний чек")
for category, revenue in sorted(revenue_by_category.items(), key=lambda x: x[1], reverse=True):
    count = paid_count_by_category[category]
    average = revenue / count if count else 0
    print(f"{category} | {revenue:.0f} | {average:.2f}")

month_count = {}
for row in rows:
    month = row["order_date"][:7]
    month_count[month] = month_count.get(month, 0) + 1
print("Заказы по месяцам")
for month, count in sorted(month_count.items()):
    print(f"{month} | {count}")


Заказы по категориям
Одежда | 1529
Электроника | 1185
Дом | 1105
Красота | 991
Книги | 653
Категория | Выручка | Средний чек
Электроника | 65270240 | 61286.61
Одежда | 18321080 | 13285.77
Дом | 10516320 | 10654.83
Красота | 5507620 | 6188.34
Книги | 2294100 | 3934.99
Заказы по месяцам
2026-09 | 2782
2026-10 | 2681


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [4]:
# Стандарт
revenue_by_month = {month: 0 for month in month_count}
for row in rows:
    if row["status"] == "оплачен":
        month = row["order_date"][:7]
        revenue_by_month[month] += to_num(row["amount"])
print("Выручка по месяцам")
for month, revenue in sorted(revenue_by_month.items()):
    print(f"{month} | {revenue:.0f}")
change = (revenue_by_month["2026-10"] - revenue_by_month["2026-09"]) / revenue_by_month["2026-09"] * 100
print(f"Изменение октября к сентябрю {change:.2f}%")

orders_by_city = {}
cancelled_by_city = {}
for row in rows:
    city = clean_city(row["city"])
    orders_by_city[city] = orders_by_city.get(city, 0) + 1
    if row["status"] == "отменён":
        cancelled_by_city[city] = cancelled_by_city.get(city, 0) + 1
cancellation_share = {}
print("Город | Доля отмен")
for city, count in sorted(orders_by_city.items()):
    cancellation_share[city] = cancelled_by_city.get(city, 0) / count * 100
    print(f"{city} | {cancellation_share[city]:.1f}%")

revenue_by_customer = {}
for row in rows:
    if row["status"] == "оплачен":
        customer = row["customer_id"]
        revenue_by_customer[customer] = revenue_by_customer.get(customer, 0) + to_num(row["amount"])
top_customers = sorted(revenue_by_customer.items(), key=lambda x: x[1], reverse=True)[:5]
anomaly_customer = top_customers[0][0]
print("Топ 5 клиентов по выручке")
for customer, revenue in top_customers:
    print(f"{customer} | {revenue:.0f}")
print("Заказы первого клиента")
for row in rows:
    if row["customer_id"] == anomaly_customer:
        print(f"{row['order_date']} | {row['category']} | {to_num(row['amount']):.0f} | {row['status']}")

customers_by_city = {}
for row in rows:
    city = clean_city(row["city"])
    if city not in customers_by_city:
        customers_by_city[city] = set()
    customers_by_city[city].add(row["customer_id"])
print("Город | Уникальные клиенты")
for city, customers in sorted(customers_by_city.items()):
    print(f"{city} | {len(customers)}")


Выручка по месяцам
2026-09 | 57489850
2026-10 | 44419510
Изменение октября к сентябрю -22.74%
Город | Доля отмен
Актобе | 10.3%
Алматы | 7.9%
Астана | 8.8%
Атырау | 7.5%
Караганда | 8.9%
Шымкент | 20.4%
Топ 5 клиентов по выручке
TEST | 29999970
C01627 | 257180
C00485 | 215920
C01338 | 192080
C02122 | 191670
Заказы первого клиента
2026-09-12 11:00 | Электроника | 9999990 | оплачен
2026-09-25 15:00 | Электроника | 9999990 | оплачен
2026-10-03 10:00 | Электроника | 9999990 | оплачен
Город | Уникальные клиенты
Актобе | 358
Алматы | 1351
Астана | 981
Атырау | 138
Караганда | 451
Шымкент | 655


На первом месте TEST у него выручка 29999970 и три заказа каждый по 9999990
Похоже это тестовый клиент а не обычный покупатель потому что имя TEST и суммы одинаковые и намного больше чем у остальных
Два заказа у него в сентябре и один в октябре это не полные дубли потому что даты и номера заказов разные


## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [5]:
# Челлендж
seen = set()
unique_rows = []
duplicate_count = 0
duplicates_by_month = {}
for row in rows:
    key = tuple(row.values())
    if key in seen:
        duplicate_count += 1
        month = row["order_date"][:7]
        duplicates_by_month[month] = duplicates_by_month.get(month, 0) + 1
    else:
        seen.add(key)
        unique_rows.append(row)
print(f"Всего дублей {duplicate_count}")
for month, count in sorted(duplicates_by_month.items()):
    print(f"{month} | {count}")

clean_revenue_by_month = {month: 0 for month in month_count}
for row in unique_rows:
    if row["status"] == "оплачен" and row["customer_id"] != anomaly_customer:
        month = row["order_date"][:7]
        clean_revenue_by_month[month] += to_num(row["amount"])
print("Выручка без дублей и тестового клиента")
for month, revenue in sorted(clean_revenue_by_month.items()):
    print(f"{month} | {revenue:.0f}")
clean_change = (clean_revenue_by_month["2026-10"] - clean_revenue_by_month["2026-09"]) / clean_revenue_by_month["2026-09"] * 100
print(f"До очистки {change:.2f}%")
print(f"После очистки {clean_change:.2f}%")
print(f"Разница {clean_change - change:.2f} процентных пункта")

customer_order_count = {}
for row in unique_rows:
    customer = row["customer_id"]
    customer_order_count[customer] = customer_order_count.get(customer, 0) + 1
max_customer, max_orders = max(customer_order_count.items(), key=lambda x: x[1])
print(f"Больше всего заказов у {max_customer} их {max_orders}")


Всего дублей 180
2026-09 | 180
Выручка без дублей и тестового клиента
2026-09 | 35243410
2026-10 | 34419520
До очистки -22.74%
После очистки -2.34%
Разница 20.40 процентных пункта
Больше всего заказов у C01818 их 10


После очистки выручка в октябре упала примерно на 2.34 процента а в начале получалось 22.74
В сентябре было 180 дублей из за них сумма за этот месяц получилась больше
Еще у клиента TEST три заказа почти по 10 миллионов и два из них были в сентябре поэтому сравнение было не правильным
Без этих строк сентябрь дает 35243410 а октябрь 34419520 поэтому падение получается намного меньше
